In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("Restaurant_Reviews.tsv", sep = "\t")
df.head()

,Review,Liked
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Review  1000 non-null   object
 1   Liked   1000 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 15.8+ KB


In [ ]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
import re

# Download required NLTK data
nltk.download('punkt')
nltk.download('stopwords')

# Initialize stop words and Porter Stemmer
stop_words = set(stopwords.words('english'))
# Remove negation words from stopwords
negation_words = {'no', 'not', 'nor', 'none', 'never', 'neither'}
stop_words = stop_words - negation_words

ps = PorterStemmer()

# Contraction mapping
contraction_map = {
    "isn't": "is not",
    "aren't": "are not",
    "wasn't": "was not",
    "weren't": "were not",
    "haven't": "have not",
    "hasn't": "has not",
    "hadn't": "had not",
    "won't": "will not",
    "wouldn't": "would not",
    "don't": "do not",
    "doesn't": "does not",
    "didn't": "did not",
    "can't": "cannot",
    "couldn't": "could not",
    "shouldn't": "should not",
    "mightn't": "might not",
    "mustn't": "must not"
}

def expand_contractions(text):
    contractions_pattern = re.compile('({})'.format('|'.join(contraction_map.keys())), 
                                    flags=re.IGNORECASE|re.DOTALL)
    def expand_match(contraction):
        match = contraction.group(0)
        return contraction_map.get(match.lower(), match)
    expanded_text = contractions_pattern.sub(expand_match, text)
    return expanded_text

# Function to preprocess text
def preprocess_text(text):
    # Handle non-string inputs (e.g., NaN or float)
    if not isinstance(text, str):
        return ''
    
    # Expand contractions
    text = expand_contractions(text)
    
    # Tokenize and convert to lowercase
    tokens = word_tokenize(text.lower())
    
    # Remove stop words (except negations) and apply Porter Stemmer
    tokens = [ps.stem(word) for word in tokens 
              if word.isalnum() and word not in stop_words]
    
    # Join tokens back into a string
    return ' '.join(tokens)

# Apply preprocessing to the 'Review' column
df['Cleaned Review'] = df['Review'].apply(preprocess_text)
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Split the data into features (X) and target (y)
X = df['Cleaned Review']
y = df['Liked']

# 2. Split into training and test sets (typically 80-20 or 70-30 split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,  # 20% for testing
    random_state=42  # for reproducibility
)

# 3. Vectorize the text data (convert text to numerical features)
vectorizer = CountVectorizer()  # you can add parameters like max_features, ngram_range, etc.
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)  # Note: only transform, not fit_transform

# 4. Initialize and train the MultinomialNB model
nb_model = MultinomialNB()
nb_model.fit(X_train_vec, y_train)

# 5. Make predictions
y_pred = nb_model.predict(X_test_vec)

# 6. Evaluate the model
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# (Optional) To see some example predictions
test_results = pd.DataFrame({
    'Review': X_test,
    'Actual': y_test,
    'Predicted': y_pred
})
print("\nSample predictions:")
print(test_results.sample(5, random_state=42))

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\sohil\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sohil\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [5]:
df[['Review','Cleaned Review']].head(10)

,Review,Cleaned Review
0,Wow... Loved this place.,wow love place
1,Crust is not good.,crust not good
2,Not tasty and the texture was just nasty.,not tasti textur nasti
3,Stopped by during the late May bank holiday of...,stop late may bank holiday rick steve recommen...
4,The selection on the menu was great and so wer...,select menu great price
5,Now I am getting angry and I want my damn pho.,get angri want damn pho
6,Honeslty it didn't taste THAT fresh.),honeslti not tast fresh
7,The potatoes were like rubber and you could te...,potato like rubber could tell made ahead time ...
8,The fries were great too.,fri great
9,A great touch.,great touch


In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Assuming df is your DataFrame
# df = pd.read_csv('your_data.csv')  # Load your data here if needed

# 1. Split data into features (X) and target (y)
X = df['Cleaned Review']
y = df['Liked']

# 2. Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,  # 20% for testing
    random_state=42  # For reproducibility
)

# 3. Vectorize text data
vectorizer = CountVectorizer(
    max_features=5000,  # Limit features to reduce noise
    ngram_range=(1, 2),  # Use unigrams and bigrams
    stop_words='english'  # Remove common English words
)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)  # Only transform, not fit

# 4. Train MultinomialNB model
nb_model = MultinomialNB(alpha=1.0)  # Smoothing parameter
nb_model.fit(X_train_vec, y_train)

# 5. Evaluate model
y_pred = nb_model.predict(X_test_vec)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# 6. Test function for new reviews
def test_review(review_text, model, vectorizer):
    """
    Test a new review and predict its sentiment (positive/negative).
    
    Parameters:
    - review_text (str): The review text to classify
    - model: Trained model (e.g., MultinomialNB)
    - vectorizer: Fitted CountVectorizer object
    
    Returns:
    - str: Prediction (Positive/Negative) with probability
    """
    # Vectorize the input review
    review_vec = vectorizer.transform([review_text])
    
    # Predict
    prediction = model.predict(review_vec)[0]
    probabilities = model.predict_proba(review_vec)[0]
    
    # Format output
    label = "Positive" if prediction == 1 else "Negative"
    prob = probabilities[prediction]
    
    return f"Review: {review_text}\nPrediction: {label} (Probability: {prob:.2%})"

# 7. Test the function with sample reviews
test_reviews = [
    "The food was amazing and the service was great!",
    "Terrible experience, the product was broken.",
    "It was okay, not too bad but not great either."
]

print("\nTest Predictions:")
for review in test_reviews:
    print(test_review(review, nb_model, vectorizer))
    print("-" * 50)

# 8. Show sample predictions from test set
test_results = pd.DataFrame({
    'Review': X_test,
    'Actual': y_test,
    'Predicted': y_pred
})
print("\nSample predictions from test set:")
print(test_results.sample(5, random_state=42))

Accuracy: 0.75

Classification Report:
              precision    recall  f1-score   support

           0       0.73      0.77      0.75        96
           1       0.78      0.73      0.75       104

    accuracy                           0.75       200
   macro avg       0.75      0.75      0.75       200
weighted avg       0.75      0.75      0.75       200


Confusion Matrix:
[[74 22]
 [28 76]]

Test Predictions:
Review: The food was amazing and the service was great!
Prediction: Positive (Probability: 98.25%)
--------------------------------------------------
Review: Terrible experience, the product was broken.
Prediction: Negative (Probability: 50.50%)
--------------------------------------------------
Review: It was okay, not too bad but not great either.
Prediction: Positive (Probability: 79.79%)
--------------------------------------------------

Sample predictions from test set:
                                            Review  Actual  Predicted
436                       

In [17]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

# Assuming df is your DataFrame
# df = pd.read_csv('your_data.csv')  # Load your data here if needed

# Basic preprocessing function
def preprocess_text(text):
    text = text.lower()  # Lowercase
    text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
    return text

# Apply preprocessing
df['Cleaned Review'] = df['Cleaned Review'].apply(preprocess_text)

# 1. Split data into features (X) and target (y)
X = df['Cleaned Review']
y = df['Liked']

# 2. Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,  # 20% for testing
    random_state=42,  # For reproducibility
    stratify=y  # Ensure balanced classes in splits
)

# 3. Create a pipeline with TfidfVectorizer and LogisticRegression
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        stop_words=stopwords.words('english'),
        ngram_range=(1, 2),  # Unigrams and bigrams
        max_features=5000,  # Limit features
        max_df=0.8,  # Ignore terms in >80% of documents
        min_df=2  # Ignore terms in <2 documents
    )),
    ('clf', LogisticRegression(
        class_weight='balanced',  # Handle imbalanced classes
        max_iter=1000
    ))
])

# 4. Hyperparameter tuning with GridSearchCV
param_grid = {
    'tfidf__max_features': [3000, 5000],
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'clf__C': [0.1, 1.0, 10.0]  # Regularization strength
}
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,  # 5-fold cross-validation
    scoring='accuracy',
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

# 5. Best model
best_model = grid_search.best_estimator_
print("Best Parameters:", grid_search.best_params_)
print("Best Cross-Validation Accuracy:", grid_search.best_score_)

# 6. Evaluate on test set
y_pred = best_model.predict(X_test)
print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# 7. Test function for new reviews
def test_review(review_text, model):
    """
    Test a new review and predict its sentiment (positive/negative).
    
    Parameters:
    - review_text (str): The review text to classify
    - model: Trained pipeline (TfidfVectorizer + LogisticRegression)
    
    Returns:
    - str: Prediction (Positive/Negative) with probability
    """
    # Preprocess the input review
    review_text = preprocess_text(review_text)
    
    # Predict
    prediction = model.predict([review_text])[0]
    probabilities = model.predict_proba([review_text])[0]
    
    # Format output
    label = "Positive" if prediction == 1 else "Negative"
    prob = probabilities[prediction]
    
    return f"Review: {review_text}\nPrediction: {label} (Probability: {prob:.2%})"

# 8. Test the function with sample reviews
test_reviews = [
    "The food was amazing and the service was great!",
    "Terrible experience, the product was broken.",
    "It was okay, not too bad but not great either."
]

print("\nTest Predictions:")
for review in test_reviews:
    print(test_review(review, best_model))
    print("-" * 50)

# 9. Show sample predictions from test set
test_results = pd.DataFrame({
    'Review': X_test,
    'Actual': y_test,
    'Predicted': y_pred
})
print("\nSample predictions from test set:")
print(test_results.sample(5, random_state=42))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\sohil\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Best Parameters: {'clf__C': 0.1, 'tfidf__max_features': 3000, 'tfidf__ngram_range': (1, 2)}
Best Cross-Validation Accuracy: 0.7662500000000001
Test Accuracy: 0.785

Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.79      0.79       100
           1       0.79      0.78      0.78       100

    accuracy                           0.79       200
   macro avg       0.79      0.79      0.78       200
weighted avg       0.79      0.79      0.78       200


Confusion Matrix:
[[79 21]
 [22 78]]

Test Predictions:
Review: the food was amazing and the service was great
Prediction: Positive (Probability: 63.34%)
--------------------------------------------------
Review: terrible experience the product was broken
Prediction: Negative (Probability: 50.81%)
--------------------------------------------------
Review: it was okay not too bad but not great either
Prediction: Positive (Probability: 51.47%)
--------------------------------------